# 01 · Exploratory Data Analysis — public datasets

**Goal:** understand the public images before splitting and training. Find imbalance, quality problems
and *shortcuts*: patterns the model could learn instead of the disease, such as image shape, background or source dataset.

**Inputs** (created by the data scripts, see `data/README.md`):
- `<data-root>/manifests/manifest_public_v1.csv`: one row per image
- `<data-root>/cache/thumbs_384/`: small copies of every `use` / `hold` image

**Outputs**
- Charts: `docs/eda/figures/` (committed; they contain no dataset photos)
- Image grids: `<data-root>/eda/grids/` (not committed; dataset licences restrict republishing)

Set `KISANSHIELD_DATA_ROOT` if your data is not in `E:/Datasets`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

DATA_ROOT = Path(os.environ.get("KISANSHIELD_DATA_ROOT", "E:/Datasets"))
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "taxonomy").is_dir())
MANIFEST = DATA_ROOT / "manifests" / "manifest_public_v1.csv"
THUMBS = DATA_ROOT / "cache" / "thumbs_384"
FIG_DIR = REPO / "docs" / "eda" / "figures"
GRID_DIR = DATA_ROOT / "eda" / "grids"
CACHE = DATA_ROOT / "cache"
for d in (FIG_DIR, GRID_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 80)
RNG = np.random.default_rng(42)

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=130, bbox_inches="tight")

m = pd.read_csv(MANIFEST, dtype={"source_label": str})
m["aspect"] = m["width"] / m["height"]
active = m[m["status"].isin(["use", "hold"])].copy()   # images we may train on
use = m[m["status"] == "use"].copy()
print(f"{len(m):,} files | use {len(use):,} | hold {(m.status == 'hold').sum():,} | exclude {(m.status == 'exclude').sum():,}")

## 1 · Overview: what happened to every file

In [ ]:
overview = m.pivot_table(index="dataset_id", columns="status", values="image_id", aggfunc="count", fill_value=0)
overview["total"] = overview.sum(axis=1)
display(overview)
display(m[m.status == "exclude"].groupby(["dataset_id", "reason"]).size().unstack(fill_value=0))

## 2 · Class balance (usable images)

Large gaps between classes make the model ignore rare classes. The *imbalance ratio* is largest ÷ smallest class within a crop.

In [ ]:
counts = use.pivot_table(index="class_id", columns="dataset_id", values="image_id", aggfunc="count", fill_value=0)
counts = counts.loc[counts.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(9, 6))
counts.plot.barh(stacked=True, ax=ax, width=0.8)
ax.set(xlabel="usable images", ylabel="", title="Usable images per class, by source dataset")
ax.legend(title="dataset", fontsize=8)
save(fig, "01_class_balance")
plt.show()

per_crop = use.groupby(["crop", "class_id"]).size()
balance = per_crop.groupby(level=0).agg(classes="count", smallest="min", largest="max")
balance["imbalance_ratio"] = (balance["largest"] / balance["smallest"]).round(1)
display(balance)

# share of each class that comes from its largest single source
dominance = (counts.max(axis=1) / counts.sum(axis=1)).rename("largest_source_share").round(2)
display(dominance.sort_values(ascending=False).to_frame())

## 3 · Held images (waiting on a decision)

In [ ]:
display(m[m.status == "hold"].groupby(["dataset_id", "source_label", "reason"]).size().rename("images").to_frame())

## 4 · Image size and shape

Very different sizes or shapes per dataset are a shortcut risk: the model can learn *shape → class*.

In [ ]:
size_stats = active.groupby("dataset_id").agg(
    images=("image_id", "count"),
    median_width=("width", "median"), median_height=("height", "median"),
    min_side_p05=("width", lambda s: np.percentile(np.minimum(s, active.loc[s.index, "height"]), 5)),
    median_aspect=("aspect", "median"),
)
display(size_stats.round(2))

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"wspace": 0.55})
sns.scatterplot(data=active, x="width", y="height", hue="dataset_id", s=8, alpha=0.4, ax=axes[0], linewidth=0)
axes[0].set(xscale="log", yscale="log", title="Image size (log scale)")
axes[0].legend(fontsize=7, markerscale=2)
order = active.groupby("dataset_id")["aspect"].median().sort_values().index
sns.boxplot(data=active, y="dataset_id", x="aspect", order=order, ax=axes[1], fliersize=1)
axes[1].set(xscale="log", title="Aspect ratio (width ÷ height)", ylabel="")
save(fig, "02_image_size_shape")
plt.show()

wheat = use[use.crop == "wheat"]
display(wheat.groupby(["class_id", "dataset_id"])["aspect"].median().unstack().round(2))

## 5 · What each class looks like

Random samples per (dataset, label). Grids are saved to `<data-root>/eda/grids/` for the agronomist.

In [ ]:
def letterbox(path, size=160):
    img = Image.open(path).convert("RGB")
    img.thumbnail((size, size))
    tile = Image.new("RGB", (size, size), (255, 255, 255))
    tile.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
    return tile

def grid(df, title, n=12, cols=6, name=None):
    sample = df.sample(min(n, len(df)), random_state=1)
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.8, rows * 1.9))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, (_, r) in zip(np.ravel(axes), sample.iterrows()):
        ax.imshow(letterbox(THUMBS / f"{r.image_id}.jpg"))
        ax.set_title(r.image_id[:8], fontsize=6)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    if name:
        fig.savefig(GRID_DIR / f"{name}.jpg", dpi=110)
    plt.show()
    plt.close(fig)

for (ds, label), g in active.groupby(["dataset_id", "source_label"]):
    cls = g["class_id"].dropna().iloc[0] if g["class_id"].notna().any() else "hold"
    grid(g, f"{ds} · {label} → {cls}  ({len(g):,} images)", name=f"{ds}__{label}")

## 6 · Colour, brightness and background

The **border** (outer 10% of the image) is mostly background. If border colour differs a lot between classes,
the model can classify by background instead of by disease.

In [ ]:
stats_path = CACHE / "color_stats.csv"

def color_stats(image_id):
    a = np.asarray(Image.open(THUMBS / f"{image_id}.jpg").convert("HSV"), dtype=np.float32)
    h, w = a.shape[:2]
    bh, bw = max(1, h // 10), max(1, w // 10)
    border = np.concatenate([a[:bh].reshape(-1, 3), a[-bh:].reshape(-1, 3), a[:, :bw].reshape(-1, 3), a[:, -bw:].reshape(-1, 3)])
    return dict(image_id=image_id, saturation=a[..., 1].mean(), value=a[..., 2].mean(),
                border_saturation=border[:, 1].mean(), border_value=border[:, 2].mean())

if stats_path.exists():
    cstats = pd.read_csv(stats_path)
else:
    cstats = pd.DataFrame([color_stats(i) for i in active["image_id"]])
    cstats.to_csv(stats_path, index=False)

active = active.merge(cstats, on="image_id", how="left")
active["group"] = active["dataset_id"].str.replace("_", " ").str[:18] + " · " + active["source_label"]

fig, axes = plt.subplots(1, 2, figsize=(15, 9), sharey=True)
order = active.sort_values(["crop", "dataset_id", "source_label"])["group"].unique()
sns.boxplot(data=active, y="group", x="border_value", order=order, ax=axes[0], fliersize=0.5)
sns.boxplot(data=active, y="group", x="border_saturation", order=order, ax=axes[1], fliersize=0.5)
axes[0].set(title="Background brightness (border V)", ylabel="")
axes[1].set(title="Background colourfulness (border S)")
save(fig, "03_background")
plt.show()

## 7 · Image quality: blur and exposure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=active, y="dataset_id", x="sharpness_256", ax=axes[0], fliersize=0.5)
axes[0].set(xscale="log", title="Sharpness (Laplacian variance at 256 px; lower = blurrier)", ylabel="")
sns.boxplot(data=active, y="dataset_id", x="brightness", ax=axes[1], fliersize=0.5)
axes[1].set(title="Brightness (0-255)", ylabel="")
save(fig, "04_quality")
plt.show()

for q in (0.01, 0.02, 0.05):
    print(f"sharpness p{int(q*100):02d}: {active.sharpness_256.quantile(q):.1f}")
blurriest = active.nsmallest(36, "sharpness_256")
display(blurriest.groupby(["dataset_id", "source_label"]).size().rename("in blurriest 36"))
grid(blurriest, "36 blurriest images", n=36, cols=9, name="quality__blurriest_36")

## 8 · Duplicates

In [ ]:
dups = pd.read_csv(DATA_ROOT / "manifests" / "duplicates_public_v1.csv")
print(f"groups: {len(dups):,} | images in groups: {dups['size'].sum():,}")
print(f"cross-dataset groups: {dups.cross_dataset.sum():,} | source-split leakage groups: {dups.cross_split.sum():,} "
      f"| label-conflict groups: {dups.label_conflict.sum():,}")
display(m[m.reason.isin(["exact_duplicate", "near_duplicate"])].groupby(["dataset_id", "source_label"]).size()
        .rename("removed").sort_values(ascending=False).head(15).to_frame())
display(dups[dups.label_conflict == 1]["labels"].value_counts().head(10).to_frame("groups"))

## 9 · Image embeddings: do images cluster by disease or by source?

Each image is turned into a 1,280-number vector by an ImageNet-pretrained **EfficientNet-B0** (no training).
t-SNE squeezes the vectors into 2-D. Points close together look alike to the network.

In [ ]:
import tensorflow as tf

emb_path = CACHE / "embeddings_effb0.npz"
if emb_path.exists():
    z = np.load(emb_path, allow_pickle=True)
    emb = pd.DataFrame(z["emb"], index=z["ids"])
else:
    model = tf.keras.applications.EfficientNetB0(include_top=False, pooling="avg", weights="imagenet")
    ids = active["image_id"].tolist()
    vecs = []
    for start in range(0, len(ids), 128):
        batch = np.stack([np.asarray(letterbox(THUMBS / f"{i}.jpg", 224)) for i in ids[start:start + 128]])
        vecs.append(model.predict(batch.astype("float32"), verbose=0))  # EfficientNet scales pixels itself
    emb = pd.DataFrame(np.concatenate(vecs), index=ids)
    np.savez_compressed(emb_path, emb=emb.values.astype("float16"), ids=np.array(ids))
print(emb.shape)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def tsne_plot(crop, per_group=300):
    df = active[active.crop == crop]
    df = df.sample(frac=1, random_state=0).groupby(["dataset_id", "source_label"]).head(per_group)
    x = PCA(50, random_state=0).fit_transform(emb.loc[df.image_id].values.astype("float32"))
    xy = TSNE(2, perplexity=35, init="pca", random_state=0).fit_transform(x)
    df = df.assign(x=xy[:, 0], y=xy[:, 1], label=df.class_id.fillna("hold:" + df.source_label))
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    sns.scatterplot(data=df, x="x", y="y", hue="label", s=10, alpha=0.7, ax=axes[0], linewidth=0)
    sns.scatterplot(data=df, x="x", y="y", hue="dataset_id", s=10, alpha=0.7, ax=axes[1], linewidth=0)
    axes[0].set(title=f"{crop}: coloured by class", xticks=[], yticks=[], xlabel="", ylabel="")
    axes[1].set(title=f"{crop}: coloured by source dataset", xticks=[], yticks=[], xlabel="", ylabel="")
    for ax in axes:
        ax.legend(fontsize=7, markerscale=2, loc="best")
    save(fig, f"05_tsne_{crop}")
    plt.show()

for crop in ["wheat", "soybean", "chickpea", "pea"]:
    tsne_plot(crop)

## 10 · Shortcut tests

Three quick checks with simple models (5-fold cross-validation, macro-F1):

1. **Metadata only:** predict the class from image width, height, aspect, brightness, sharpness and file size (no pixels).
   A high score means the class can be guessed *without looking at the disease*.
2. **Source identification:** within one class that comes from two datasets, predict *which dataset* from the embedding.
   Near-perfect means the sources look systematically different, so test results on one source won't transfer.
3. **Linear probe:** predict the class from the embedding. This is an optimistic ceiling for how separable the classes are.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cv = StratifiedKFold(5, shuffle=True, random_state=0)
meta_cols = ["width", "height", "aspect", "brightness", "sharpness_256", "file_bytes"]
results = []

for crop, df in use.groupby("crop"):
    if df.class_id.nunique() < 2:
        continue
    y = df.class_id
    chance = y.value_counts(normalize=True).max()
    meta = cross_val_score(RandomForestClassifier(300, n_jobs=-1, random_state=0), df[meta_cols], y, cv=cv, scoring="f1_macro").mean()
    probe = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5)),
                            emb.loc[df.image_id].values.astype("float32"), y, cv=cv, scoring="f1_macro").mean()
    results.append(dict(test="class from metadata only", scope=crop, macro_f1=meta, note=f"majority-class share {chance:.2f}"))
    results.append(dict(test="class from embedding (linear probe)", scope=crop, macro_f1=probe, note=""))

for cls, df in use.groupby("class_id"):
    if df.dataset_id.nunique() < 2:
        continue
    y = df.dataset_id
    f1 = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5)),
                         emb.loc[df.image_id].values.astype("float32"), y, cv=cv, scoring="f1_macro").mean()
    results.append(dict(test="source dataset from embedding", scope=cls, macro_f1=f1,
                        note=", ".join(f"{k} {v}" for k, v in y.value_counts().items())))

shortcut = pd.DataFrame(results).round(3)
shortcut.to_csv(FIG_DIR.parent / "shortcut_tests.csv", index=False)
display(shortcut)

## 11 · Findings

Findings, their impact and the resulting decisions are written up in `docs/eda/EDA_FINDINGS.md`.
Decisions are logged in `data/review/DECISIONS.md`.